# Projeto Avaliativo - Camada Gold

Este notebook responde às perguntas de negócio com SQL, JOINs, GROUP BY, tabelas e gráficos.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sqlalchemy import text
from banco import criar_engine, testar_conexao

pd.set_option('display.float_format', '{:,.2f}'.format)

testar_conexao()
engine = criar_engine()

## 1. Os 5 órgãos com maior custo total

In [ ]:
sql_1 = '''SELECT nome_orgao_superior, SUM(valor_total) AS custo_total FROM silver_viagem GROUP BY nome_orgao_superior ORDER BY custo_total DESC LIMIT 5;'''
df_1 = pd.read_sql(sql_1, engine)
df_1

In [ ]:
ax = df_1.plot(kind='bar', x='nome_orgao_superior', y='custo_total', legend=False, figsize=(10,5))
ax.set_title('Os 5 órgãos com maior custo total')
ax.set_xlabel('nome_orgao_superior')
ax.set_ylabel('custo_total')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 2. Os 3 destinos com maior custo médio por viagem

In [ ]:
sql_2 = '''SELECT t.destino_uf, t.destino_cidade, AVG(v.valor_total) AS custo_medio_viagem, COUNT(DISTINCT v.id_viagem) AS quantidade_viagens FROM silver_viagem v JOIN silver_trecho t ON v.id_viagem = t.id_viagem WHERE t.destino_cidade IS NOT NULL GROUP BY t.destino_uf, t.destino_cidade HAVING COUNT(DISTINCT v.id_viagem) >= 3 ORDER BY custo_medio_viagem DESC LIMIT 3;'''
df_2 = pd.read_sql(sql_2, engine)
df_2

## 3. A viagem de maior duração e seu custo total

In [ ]:
sql_3 = '''SELECT id_viagem, num_proposta, nome_orgao_superior, destinos, duracao_dias, valor_total FROM silver_viagem ORDER BY duracao_dias DESC, valor_total DESC LIMIT 1;'''
df_3 = pd.read_sql(sql_3, engine)
df_3

## 4. Tipo de pagamento com maior valor médio

In [ ]:
sql_4 = '''SELECT tipo_pagamento, AVG(valor) AS valor_medio, COUNT(*) AS quantidade_pagamentos FROM silver_pagamento GROUP BY tipo_pagamento ORDER BY valor_medio DESC LIMIT 1;'''
df_4 = pd.read_sql(sql_4, engine)
df_4

## 5. Meio de transporte mais usado nos trechos

In [ ]:
sql_5 = '''SELECT meio_transporte, COUNT(*) AS quantidade_trechos FROM silver_trecho WHERE meio_transporte IS NOT NULL GROUP BY meio_transporte ORDER BY quantidade_trechos DESC LIMIT 5;'''
df_5 = pd.read_sql(sql_5, engine)
df_5

In [ ]:
ax = df_5.plot(kind='bar', x='meio_transporte', y='quantidade_trechos', legend=False, figsize=(10,5))
ax.set_title('Meio de transporte mais usado nos trechos')
ax.set_xlabel('meio_transporte')
ax.set_ylabel('quantidade_trechos')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 6. UF de destino que aparece em mais trechos

In [ ]:
sql_6 = '''SELECT destino_uf, COUNT(*) AS quantidade_trechos FROM silver_trecho WHERE destino_uf IS NOT NULL GROUP BY destino_uf ORDER BY quantidade_trechos DESC LIMIT 10;'''
df_6 = pd.read_sql(sql_6, engine)
df_6

In [ ]:
ax = df_6.plot(kind='bar', x='destino_uf', y='quantidade_trechos', legend=False, figsize=(10,5))
ax.set_title('UF de destino que aparece em mais trechos')
ax.set_xlabel('destino_uf')
ax.set_ylabel('quantidade_trechos')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 7. Órgão que pagou mais no total

In [ ]:
sql_7 = '''SELECT nome_orgao_pagador, SUM(valor) AS total_pago FROM silver_pagamento WHERE nome_orgao_pagador IS NOT NULL GROUP BY nome_orgao_pagador ORDER BY total_pago DESC LIMIT 5;'''
df_7 = pd.read_sql(sql_7, engine)
df_7

In [ ]:
ax = df_7.plot(kind='bar', x='nome_orgao_pagador', y='total_pago', legend=False, figsize=(10,5))
ax.set_title('Órgão que pagou mais no total')
ax.set_xlabel('nome_orgao_pagador')
ax.set_ylabel('total_pago')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## Criação das tabelas Gold

In [ ]:
with open('3_gold_consultas.sql', 'r', encoding='utf-8') as f:
    sql_gold = f.read()

with engine.begin() as conn:
    conn.execute(text(sql_gold))

print('Consultas e tabelas Gold executadas com sucesso.')